In [2]:
import os
import json
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
PREDSDIR   = CONFIGS['filepaths']['predictions']
RAWDIR     = CONFIGS['filepaths']['raw']
MODELS     = CONFIGS['experiments']
LATRANGE   = CONFIGS['domain']['latrange']
LONRANGE   = CONFIGS['domain']['lonrange']
FIELDVARS  = MODELS['sr']['runs']['sr_gauss']['fieldvars']
NNSEEDS    = MODELS['nn']['seeds']
ORDER      = ['pod_bl','nn_bl','nn_full','nn_nonparam','nn_gauss','sr_lo','sr_bl','sr_med','sr_hi']
SPLIT      = 'test'
NBINS      = 20
MINSAMPLES = 50

COLORS = {}
LABELS = {}
for name,config in {**MODELS['pod']['runs'],**MODELS['nn']['runs'],**MODELS['sr']['optimizedeqs']}.items():
    COLORS[name] = config['color']
    LABELS[name] = config['description']

In [4]:
def kernel_integrate(fields,weights,dsig,mask=None):
    w = fields*weights[None,:,:]*dsig[None,None,:]
    if mask is not None: w = w*mask[:,None,:]
    return w.sum(axis=2)

def r2(ytrue,ypred):
    mask = np.isfinite(ytrue)&np.isfinite(ypred)
    ytrue,ypred = ytrue[mask],ypred[mask]
    return 1-np.sum((ytrue-ypred)**2)/np.sum((ytrue-ytrue.mean())**2)

In [ ]:
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    stats = json.load(f)
tpmean = float(stats['tp_mean'])
tpstd  = float(stats['tp_std'])

def flatten(da):
    if 'time' in da.dims: return da.transpose('time','lat','lon').values.ravel()
    return np.tile(da.values,(ntime,1,1)).ravel()

with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime,nlat,nlon = ds.sizes['time'],ds.sizes['lat'],ds.sizes['lon']
    nsig = ds.sizes.get('sig',1)
    lat  = ds['lat'].values
    lon  = ds['lon'].values
    dsig = ds['dsig'].values
    fields   = np.stack([ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for v in FIELDVARS],axis=1)
    surfmask = (ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1,nsig) if 'surfmask' in ds else None)

kernels = []
for seed in NNSEEDS:
    wpath = os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc')
    if os.path.exists(wpath):
        with xr.open_dataset(wpath,engine='h5netcdf') as ds:
            kernels.append(ds['k'].values)
ki = (np.mean([kernel_integrate(fields,k,dsig,surfmask) for k in kernels],axis=0) if kernels else fields.mean(axis=2))
rhk,thetaek,thetaestark = ki[:,0],ki[:,1],ki[:,2]

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    obsflat = ds.tp.transpose('time','lat','lon').values.ravel()
    sdoraw  = flatten(ds['sdo'])

semap  = seflat = None
sepath = os.path.join(RAWDIR,'ERA5_surface_elevation.nc')
if os.path.exists(sepath):
    with xr.open_dataset(sepath,engine='netcdf4') as ds:
        sevar = list(ds.data_vars)[0]
        se = ds[sevar]
        if 'time' in se.dims:
            se = se.isel(time=0)
        se    = se.interp(lat=lat,lon=lon,method='linear')
        semap = se.values.astype(np.float32)
    seflat = np.tile(semap,(ntime,1,1)).ravel()
sdomap = sdoraw.reshape(ntime,nlat,nlon)[0]

In [6]:
def load_pred(name):
    path = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
    if not os.path.exists(path): return None
    with xr.open_dataset(path,engine='h5netcdf') as ds:
        da = ds['tp'].load()
    if 'seed' in da.dims: da = da.mean('seed')
    if 'complexity' in da.dims: da = da.isel(complexity=0)
    pred = da.transpose('time','lat','lon').values.ravel()
    return pred if pred.shape[0]==obsflat.shape[0] else None

MODELPRED = {name: p for name in ORDER if (p:=load_pred(name)) is not None}
valid     = np.isfinite(obsflat)
print(f'Loaded {len(MODELPRED)} models')
for name,pred in MODELPRED.items():
    print(f'  {LABELS.get(name,name):20s}  R²={r2(obsflat[valid],pred[valid]):.3f}')

Loaded 0 models


In [7]:
def to_map(flat):
    return np.nanmean(flat.reshape(ntime,nlat,nlon),axis=0)

obsmap    = to_map(obsflat)
residmaps = {name:to_map(MODELPRED[name])-obsmap for name in MODELPRED}

In [ ]:
names     = [n for n in ORDER if n in residmaps]
allpanels = ['obs']+names
ncols = min(3,len(allpanels)); nrows = -(-len(allpanels)//ncols)
kwmap = dict(coast=True,latlim=LATRANGE,lonlim=LONRANGE,
             latlines=[10,15,20],lonlines=[65,75,85],grid=False)
fig,axs = pplt.subplots(nrows=nrows,ncols=ncols,proj='cyl',refwidth=2.5,share=False)
axsf    = np.atleast_1d(axs).ravel()
mc = mr = None
for i,(ax,panel) in enumerate(zip(axsf,allpanels)):
    row,col = divmod(i,ncols)
    if panel=='obs':
        mc = ax.pcolormesh(lon,lat,obsmap,cmap='Blues',vmin=0,vmax=2,extend='max')
        ax.format(title='Observed')
    else:
        mr = ax.pcolormesh(lon,lat,residmaps[panel],cmap='DryWet',vmin=-0.5,vmax=0.5,extend='both')
        ax.format(title=LABELS.get(panel,panel))
    ax.format(lonlabels='b' if row==nrows-1 or i>=len(allpanels)-ncols else False,
              latlabels='l' if col==0 else False,**kwmap)
for ax in axsf[len(allpanels):]: ax.set_visible(False)
if mc: fig.colorbar(mc,loc='b',label='Total Precipitation (mm)',col=1)
if mr: fig.colorbar(mr,loc='b',label='Predicted − Observed (mm)',col=(2,ncols))
pplt.show()

In [ ]:
srnames = [n for n in ['sr_bl','sr_lo','sr_med','sr_hi'] if n in residmaps]
if srnames:
    kwmap = dict(coast=True,latlim=LATRANGE,lonlim=LONRANGE,
                 latlines=[10,15,20],lonlines=[65,75,85],grid=False)
    fig,axs = pplt.subplots(nrows=1,ncols=len(srnames),proj='cyl',refwidth=2.5,share=False)
    axsf    = np.atleast_1d(axs).ravel()
    m = None
    for i,(ax,name) in enumerate(zip(axsf,srnames)):
        m = ax.pcolormesh(lon,lat,residmaps[name],cmap='DryWet',vmin=-0.5,vmax=0.5,extend='both')
        ax.contour(lon,lat,sdomap,levels=[100,300,500],colors='k',linewidths=0.5)
        ax.format(title=LABELS.get(name,name),
                  latlabels='l' if i==0 else False,lonlabels='b',**kwmap)
    if m: fig.colorbar(m,loc='b',label='Predicted − Observed (mm)')
    pplt.show()

In [ ]:
panels = [('SDO (m)',sdomap)]
if semap is not None:
    panels.append(('Surface Elevation (m)',semap))
kwmap = dict(coast=True,latlim=LATRANGE,lonlim=LONRANGE,
             latlines=[10,15,20],lonlines=[65,75,85],grid=False)
fig,axs = pplt.subplots(nrows=1,ncols=len(panels),proj='cyl',refwidth=3,share=False)
axsf    = np.atleast_1d(axs).ravel()
m = None
for i,(ax,(title,arr)) in enumerate(zip(axsf,panels)):
    m = ax.pcolormesh(lon,lat,arr,cmap='terrain',vmin=0,vmax=900,extend='max')
    ax.format(title=title,latlabels='l' if i==0 else False,lonlabels='b',**kwmap)
fig.colorbar(m,loc='b',label='Elevation (m)')
pplt.show()

In [ ]:
namesorog = [n for n in ['sr_hi','nn_gauss','sr_med','nn_bl'] if n in MODELPRED]
obs       = obsflat[valid]

orogpanels = [('SDO (m)',sdoraw)]
if seflat is not None:
    orogpanels.append(('Surface Elevation (m)',seflat))

fig,axs = pplt.subplots(nrows=1,ncols=len(orogpanels),refwidth=3,refheight=2,share=False)
axsf    = np.atleast_1d(axs).ravel()
for ax,(oroglabel,orogflat) in zip(axsf,orogpanels):
    orog  = orogflat[valid]
    lo,hi = np.percentile(orog,[2,98])
    edges = np.linspace(lo,hi,11); xc = 0.5*(edges[:-1]+edges[1:])
    idxs  = np.clip(np.digitize(orog,edges)-1,0,9)
    for name in namesorog:
        pred = MODELPRED[name][valid]
        r2s  = [r2(obs[idxs==i],pred[idxs==i]) if (idxs==i).sum()>=500 else np.nan
                for i in range(10)]
        ax.plot(xc,r2s,color=COLORS[name],linewidth=1.5,marker='o',markersize=3,
                label=LABELS[name])
    ax.format(xlabel=oroglabel,ylabel=r'$R^2$',title=f'Skill vs. {oroglabel}',grid=False)
    ax.legend(loc='ur',ncols=1)
pplt.show()